In [1]:
class EndOfMessages(Exception):
    pass


class FakeWebSocket:
    def __init__(self, incoming: list[str]):
        self.incoming = list(incoming)
        self.calls: list[str] = []

    async def accept(self) -> None:
        self.calls.append("accept")

    async def receive_text(self) -> str:
        self.calls.append("receive_text")
        if not self.incoming:
            raise EndOfMessages()
        return self.incoming.pop(0)

In [3]:
async def observe_lifecycle(ws: FakeWebSocket) -> list[str]:
    await ws.accept()
    ws.calls.append("heartbeat.start")
    ws.calls.append("presence.connect")
    try:
        while True:
            text = await ws.receive_text()
            ws.calls.append(f"message:{text}")
    except EndOfMessages:
        ws.calls.append("disconnect")
    finally:
        ws.calls.append("heartbeat.stop")
        ws.calls.append("subscriber.detach")
        ws.calls.append("presence.disconnect")
    return ws.calls

fake_ws = FakeWebSocket(["first", "second"])
calls = await observe_lifecycle(fake_ws)
calls

['accept',
 'heartbeat.start',
 'presence.connect',
 'receive_text',
 'message:first',
 'receive_text',
 'message:second',
 'receive_text',
 'disconnect',
 'heartbeat.stop',
 'subscriber.detach',
 'presence.disconnect']